####1. Read data from the sales_sample.csv file and analyse to identify problems

1.1 Define schema

In [0]:
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

1.2 Read data

In [0]:
sales_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(file_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv")
)

sales_raw_df.display()

id,name,dop,phone,amount,discount
100,Prashant,2020-06-15,9238614990,12000,18.5
101,David,2018-08-7,8908617610,15000,nil
102,Simran,14-05-2019,null,3000000000,21


1.3 Describe the data

In [0]:
sales_raw_df.describe().display()

1.4 List down the problems you want to fix
1. Convert id from integer to string and rename it as transaction_id.
2. Rename the name column to customer_name.
3. Convert the dop to date format and rename the column to date_of_purchase.
4. Rename the phone column to customer_phone
5. Convert the amount to a long value and filter out nulls and outlier values. 
6. Rename the column to purchase_amount
7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

####2. Prepare and clean the Dataframe using appropriate transformations

2.1 Transform

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

from pyspark.sql.functions import to_date

df2 = sales_raw_df.withColumn("id",sales_raw_df["id"].cast(StringType()))
df2 = df2.withColumnRenamed("name","customer_name")
df2 = df2.withColumnRenamed("phone","customer_phone")

df3= df2.selectExpr("id as transaction_id","customer_name","dop","cast(amount as long) as purchase_amount","discount")

display(df3)
# df2= df2.selectExpr("customer_name","date_of_purchase","cast(discount as double) as applied_discount","purchase_amount","transaction_id")

# df2 =df2.na.fill(value=0,subset=["applied_discount"]).show()
# display(df2)

 #nil we cannot cast as previous stpes fails

 #try cast is use for eg 18.5 we can cast to double, for nil it will fail so we use try_cast
 #here try cast converting nil it will return instead of null and top of that we apply nvl
 #converting nil and null values to zero

df3 = df3.selectExpr("customer_name",
                     "nvl(try_cast(dop as date), to_date(dop,'dd-MM-yyyy')) as date_of_purchase",
                     "nvl(try_cast(discount as double),0) as applied_discount",
                     "purchase_amount",
                     "transaction_id").filter("purchase_amount is not null")
display(df3)

# here date values are wrong




transaction_id,customer_name,dop,purchase_amount,discount
100,Prashant,2020-06-15,12000,18.5
101,David,2018-08-7,15000,nil
102,Simran,14-05-2019,3000000000,21


customer_name,date_of_purchase,applied_discount,purchase_amount,transaction_id
Prashant,2020-06-15,18.5,12000,100
David,2018-08-07,0.0,15000,101
Simran,2019-05-14,21.0,3000000000,102


2.2 Verify statistics

In [0]:
sales_df.describe("purchase_amount", "applied_discount").display()

&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>

